In [10]:
import numpy as np     
import pandas as pd   
from sklearn.preprocessing import LabelEncoder

In [12]:
temp_df=pd.read_csv('housing_model_ready_after_outlier_treatment.csv')
df=temp_df.copy()


In [23]:
pd.set_option('display.max_columns',None)

In [5]:
bool_cols = [col for col in df.columns if df[col].dtype == 'bool' or df[col].dtype == 'object']

In [7]:
print(df.dtypes[df.dtypes == 'object'])  # Should be empty
print(df.select_dtypes(include='bool').columns.tolist())  # Should be empty

category         object
title            object
district         object
neighborhood     object
location_raw     object
facing           object
price_segment    object
dtype: object
['is_under_construction', 'is_wide_road', 'is_commercial_suspect', 'is_area_estimated']


In [13]:
# --- STEP 1: DROP UNWANTED COLUMNS ---
# price_per_aana is dropped to prevent data leakage (it contains the answer)
# log_price is dropped as it's a target, not a feature
cols_to_drop = ['title', 'location_raw', 'built_year_ad', 
                'is_commercial_suspect', 'price_per_aana', 'log_price']
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [14]:
# --- STEP 2: LOG TRANSFORMS (Handling Skewness) ---
df['log_land'] = np.log1p(df['land_area_aana'])
df['log_build_up'] = np.log1p(df['build_up_area'])

In [15]:
# --- STEP 3: ENCODE CATEGORICALS ---
le = LabelEncoder()

In [25]:
# Category: Encode 'house' vs 'land' (Important for distinguishing type)
df['category'] = df['category'].map({'house': 1, 'land': 0}).fillna(0).astype(int)

In [17]:
# District & Facing: Simple Label Encoding
df['district'] = le.fit_transform(df['district'].astype(str))
df['facing'] = le.fit_transform(df['facing'].astype(str))

In [18]:
# Price Segment: Ordinal Encoding
segment_map = {'Budget': 1, 'Mid-Range': 2, 'High-End': 3, 'Ultra-Luxury': 4}
df['price_segment'] = df['price_segment'].map(segment_map)

In [19]:
# Neighborhood: Target Encoding (Mean of Log-Price)
target_means = df.groupby('neighborhood')['total_price'].apply(lambda x: np.log1p(x).mean())
global_log_mean = np.log1p(df['total_price']).mean()
df['neighborhood_encoded'] = df['neighborhood'].map(target_means).fillna(global_log_mean)
df.drop(columns=['neighborhood'], inplace=True)

In [20]:
bool_cols = df.select_dtypes(include=['bool', 'object']).columns.tolist()
for col in bool_cols:
    try:
        df[col] = df[col].astype(int)
    except (ValueError, TypeError):
        print(f"⚠️ Could not convert {col} — check this column manually")

In [27]:
# --- STEP 5: FINAL VERIFICATION ---
print("=== Feature Engineering Verification ===")
non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
if not non_numeric:
    print("✅ All features (including Category) are now numeric.")
else:
    print(f"⚠️ Non-numeric columns remaining: {non_numeric}")

print(f"✅ Final Shape: {df.shape}")

=== Feature Engineering Verification ===
✅ All features (including Category) are now numeric.
✅ Final Shape: (2506, 29)


In [28]:
df.sample(12)

,category,district,total_price,land_area_aana,build_up_area,floors,facing,road_width_feet,bedrooms,bathrooms,parking_cars,parking_bikes,house_age,is_under_construction,amenity_count,has_modular_kitchen,has_parquet,has_drainage,has_solar,has_parking,has_garden,is_wide_road,is_area_estimated,luxury_score,price_segment,is_incomplete_listing,log_land,log_build_up,neighborhood_encoded
365,0,2,38000000.0,4.0,3354.05,3.5,4,20.0,6.0,4.0,0.0,0.0,4.0,0,4,0,0,1,0,1,0,0,1,2,3,0,1.609438,8.118222,17.145675
1286,0,1,34000000.0,5.0,2994.69,2.5,2,13.0,6.0,5.0,0.0,0.0,8.0,0,5,0,0,1,0,1,0,0,1,2,3,0,1.791759,8.004930,17.364400
45,0,2,60000000.0,7.0,4192.56,2.5,0,13.0,5.0,3.0,2.0,3.0,6.0,0,11,0,1,0,0,1,0,0,1,2,4,0,2.079442,8.341305,17.374887
158,0,1,24000000.0,3.2,1916.60,2.5,0,13.0,6.0,5.0,0.0,0.0,12.0,0,11,1,1,0,0,1,0,0,1,3,1,0,1.435085,7.558830,17.127235
651,0,2,25000000.0,3.1,1856.71,2.5,1,13.0,6.0,3.0,0.0,0.0,26.0,0,4,0,0,1,0,1,0,0,1,2,1,0,1.410987,7.527100,16.948366
2043,0,1,22000000.0,3.3,2767.09,3.5,1,13.0,5.0,4.0,0.0,0.0,22.0,0,4,0,0,1,0,1,0,0,1,2,1,0,1.458615,7.925913,17.098501
2224,0,1,37000000.0,6.0,3593.62,2.5,5,15.0,5.0,5.0,0.0,0.0,6.0,0,0,0,0,0,0,0,0,0,1,0,3,0,1.945910,8.187194,17.711204
247,0,2,28500000.0,3.0,1796.81,2.5,0,13.0,9.0,4.0,0.0,0.0,10.0,0,11,1,1,0,0,1,0,0,1,3,2,0,1.386294,7.494325,16.948366
1897,0,1,45000000.0,5.0,2994.69,2.5,0,18.0,6.0,4.0,0.0,0.0,11.0,0,7,0,1,1,0,1,1,0,1,4,3,0,1.791759,8.004930,17.527442
1691,0,1,18500000.0,3.2,1916.60,2.5,7,13.0,6.0,4.0,0.0,0.0,23.0,0,5,0,1,1,0,1,0,0,1,3,1,0,1.435085,7.558830,16.780968


In [29]:
df.to_csv('housing_features_ready_after_feature_engineering.csv', index=False)
print(f"✅ Saved — Shape: {df.shape}")

✅ Saved — Shape: (2506, 29)
